In [1]:
# 커널 확인
print('hello')

hello


In [2]:
import os
import re

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

# =========================================================
# 0. 경로 설정
# =========================================================
input_csv = (
    'C:/potenup3/pj02_Daangn-marke/data/csv/merged_dedup_siglip2_labeled_clean.csv'
)

output_dir = 'C:/potenup3/pj02_Daangn-marke/data/processed'
os.makedirs(output_dir, exist_ok=True)

output_train_csv = os.path.join(output_dir, 'train_feature_v21.csv')
output_valid_csv = os.path.join(output_dir, 'valid_feature_v21.csv')
output_test_csv = os.path.join(output_dir, 'test_feature_v21.csv')

output_train_parquet = os.path.join(output_dir, 'train_feature_v21.parquet')
output_valid_parquet = os.path.join(output_dir, 'valid_feature_v21.parquet')
output_test_parquet = os.path.join(output_dir, 'test_feature_v21.parquet')


# =========================================================
# 1. 유틸 함수
# =========================================================
# 텍스트 안전 보정: 결측 -> 빈 문자열, 문자열화, 양쪽 공백 제거
def safe_text(series: pd.Series) -> pd.Series:
    return series.fillna('').astype(str).str.strip()


# 숫자 안전 보정: numeric이면 그대로, 아니면 숫자만 추출 후 numeric으로 변환
def safe_numeric(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors='coerce')
    s = series.astype(str).str.replace(r'[^\d\.\-]', '', regex=True).replace('', np.nan)
    return pd.to_numeric(s, errors='coerce')


# 날짜 안전 보정: to_datetime으로 변환, 실패하면 NaT
def parse_datetime(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, errors='coerce')


# 단어 수 세기: 공백 기준으로 단어 수 계산, 빈 문자열은 0
def count_words(text: str) -> int:
    if not isinstance(text, str) or text.strip() == '':
        return 0
    return len(re.findall(r'\S+', text))


# 텍스트 안에 keywords 중 하나라도 포함되면 1, 아니면 0
def contains_any_keyword(text: str, keywords: list[str]) -> int:
    """
    text 안에 keywords 중 하나라도 포함되면 1
    """
    if not isinstance(text, str):
        return 0
    text_lower = text.lower()
    for kw in keywords:
        if kw in text_lower:
            return 1
    return 0


# top3_labels에서 첫 번째 라벨 추출
def parse_top1_label(x: str) -> str:
    try:
        parts = [p.strip() for p in str(x).split(',') if p.strip() != '']
        return parts[0] if len(parts) >= 1 else 'unknown'
    except Exception:
        return 'unknown'


# top3_scores에서 첫 번째 점수 추출
def parse_top1_score(x: str) -> float:
    try:
        vals = [float(v.strip()) for v in str(x).split(',') if v.strip() != '']
        return vals[0] if len(vals) >= 1 else np.nan
    except Exception:
        return np.nan


# top3_scores에서 첫 번째 점수와 두 번째 점수의 차이 계산
def parse_top1_top2_gap(x: str) -> float:
    try:
        vals = [float(v.strip()) for v in str(x).split(',') if v.strip() != '']
        if len(vals) >= 2:
            return vals[0] - vals[1]
        elif len(vals) == 1:
            return vals[0]
        else:
            return np.nan
    except Exception:
        return np.nan


# 가격 버킷화: 0-1만, 1-3만, 3-5만, 5-10만, 10만+
def bucketize_price(price: float) -> str:
    if pd.isna(price):
        return 'unknown'
    if price <= 10000:
        return '0_1만'
    elif price <= 30000:
        return '1_3만'
    elif price <= 50000:
        return '3_5만'
    elif price <= 100000:
        return '5_10만'
    else:
        return '10만+'


# 판매자 온도 버킷화: 0-30, 30-40, 40-50, 50+
def bucketize_seller_temp(temp: float) -> str:
    if pd.isna(temp):
        return 'unknown'
    if temp <= 30:
        return '0_30'
    elif temp <= 40:
        return '30_40'
    elif temp <= 50:
        return '40_50'
    else:
        return '50+'


# 특정 value가 sorted_arr 내에서 몇 퍼센트 위치하는지 계산
def percentile_rank_within_group(value: float, sorted_arr: np.ndarray) -> float:
    if pd.isna(value) or sorted_arr is None or len(sorted_arr) == 0:
        return 0.5
    pos = np.searchsorted(sorted_arr, value, side='right')
    return pos / len(sorted_arr)


# Parquet 저장 시도, 실패하면 CSV만 저장
def save_with_parquet_fallback(df: pd.DataFrame, csv_path: str, parquet_path: str):
    df.to_csv(csv_path, index=False, encoding='utf-8-sig')
    print(f'[INFO] CSV saved -> {csv_path}')

    parquet_saved = False
    for engine in ['pyarrow', 'fastparquet']:
        try:
            df.to_parquet(parquet_path, index=False, engine=engine)
            print(f'[INFO] Parquet saved -> {parquet_path} (engine={engine})')
            parquet_saved = True
            break
        except Exception as e:
            print(f'[WARN] Parquet save failed with engine={engine}: {e}')

    if not parquet_saved:
        print('[WARN] Parquet 저장 실패. CSV만 저장되었습니다.')


# =========================================================
# 2. 데이터 로드
# =========================================================
print('[INFO] Loading data...')
df = pd.read_csv(input_csv, low_memory=False)
print('[INFO] Raw shape:', df.shape)

required_cols = [
    'id',
    'price',
    'title',
    'content',
    'status',
    'createdAt',
    'sellerTemperature',
    'imageCount',
    'label',
    'coarse_label',
    'label_conf',
    'region_id',
    'brandName',
    'top3_labels',
    'top3_scores',
]
missing_cols = [c for c in required_cols if c not in df.columns]
if missing_cols:
    raise ValueError(f'필수 컬럼이 없습니다: {missing_cols}')


# =========================================================
# 3. 기본 전처리
# - clean.csv라고 하셨으므로 결측 제거는 끝난 상태 가정
# - dtype 및 안전 보정만 수행
# =========================================================
df['price'] = safe_numeric(df['price'])
df['title'] = safe_text(df['title'])
df['content'] = safe_text(df['content'])
df['status'] = safe_text(df['status'])

df['sellerTemperature'] = safe_numeric(df['sellerTemperature'])
df['imageCount'] = safe_numeric(df['imageCount'])
df['label_conf'] = safe_numeric(df['label_conf'])

df['label'] = safe_text(df['label']).replace('', 'unknown').str.lower()
df['coarse_label'] = safe_text(df['coarse_label']).replace('', 'unknown').str.lower()

# region_id는 categorical로 쓰기 위해 문자열화
df['region_id'] = df['region_id'].astype(str).fillna('unknown').replace('', 'unknown')

# brandName 정리
df['brandName'] = safe_text(df['brandName']).str.lower()
df['brandName'] = df['brandName'].replace('', 'unknown')

df['top3_labels'] = safe_text(df['top3_labels']).str.lower()
df['top3_scores'] = safe_text(df['top3_scores'])

df['createdAt'] = parse_datetime(df['createdAt'])

# 핵심 결측 혹시 남아있으면 제거
df = df.dropna(subset=['price', 'createdAt']).copy()

# 타깃 생성
df['sold'] = df['status'].isin(['Closed', 'Reserved']).astype(int)

print('[INFO] After basic cleaning:', df.shape)
print('[INFO] Sold ratio:', df['sold'].mean())


# =========================================================
# 4. train / valid / test split
# - 80 / 10 / 10
# =========================================================
# stratify 옵션으로 sold 비율 유지하면서 split
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=42, stratify=df['sold']
)

# temp_df를 다시 valid/test로 50/50 split
valid_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=42, stratify=temp_df['sold']
)

print('[INFO] Train shape:', train_df.shape)
print('[INFO] Valid shape:', valid_df.shape)
print('[INFO] Test shape :', test_df.shape)


# =========================================================
# 5. train 기준 통계/사전 생성
# =========================================================

# 5-1. brandName 상위 30개만 유지
brand_series = (
    train_df['brandName'].fillna('unknown').astype(str).str.lower().str.strip()
)
top30_brands = brand_series.value_counts().head(30).index.tolist()
top30_brands = [b for b in top30_brands if b != 'unknown']
top30_brands_set = set(top30_brands)

print('[INFO] Top 30 brands:', top30_brands)

# 5-2. label별 중앙 가격
label_median_price_map = train_df.groupby('label')['price'].median().to_dict()
global_label_median_price = train_df['price'].median()

# 5-3. region별 집계
region_post_count_map = train_df['region_id'].value_counts().to_dict()
region_sales_rate_map = train_df.groupby('region_id')['sold'].mean().to_dict()
region_price_median_map = train_df.groupby('region_id')['price'].median().to_dict()

global_region_post_count = train_df['region_id'].value_counts().median()
global_region_sales_rate = train_df['sold'].mean()
global_region_price_median = train_df['price'].median()

# 5-4. region별 price rank 계산용 정렬 배열
region_sorted_price_map = {}
for rid, grp in train_df.groupby('region_id'):
    arr = np.sort(grp['price'].dropna().values.astype(float))
    if len(arr) > 0:
        region_sorted_price_map[rid] = arr

# 5-5. 제목 상태 키워드
condition_keywords = [
    '새상품',
    '미사용',
    '미착용',
    '미개봉',
    '상태좋음',
    '상태 좋음',
    '하자없음',
    '하자 없음',
    '거의새',
    '거의 새',
    '사용감 적음',
    '오염 없음',
]


# =========================================================
# 6. feature 생성 함수
# =========================================================
def transform_features(frame: pd.DataFrame) -> pd.DataFrame:
    f = frame.copy()

    # -------------------------
    # target + key
    # -------------------------
    f['id'] = f['id']
    f['sold'] = f['sold']

    # -------------------------
    # 가격 Feature
    # -------------------------
    f['feat_price'] = f['price']
    f['feat_log_price'] = np.log1p(f['price'])
    f['feat_price_bucket'] = f['price'].apply(bucketize_price)

    f['tmp_label_median_price'] = (
        f['label'].map(label_median_price_map).fillna(global_label_median_price)
    )
    f['feat_price_vs_label_median'] = f['price'] / f['tmp_label_median_price'].replace(
        0, np.nan
    )
    f['feat_price_vs_label_median'] = (
        f['feat_price_vs_label_median'].replace([np.inf, -np.inf], np.nan).fillna(1.0)
    )

    f['tmp_region_price_median'] = (
        f['region_id'].map(region_price_median_map).fillna(global_region_price_median)
    )
    f['feat_price_vs_region_median'] = f['price'] / f[
        'tmp_region_price_median'
    ].replace(0, np.nan)
    f['feat_price_vs_region_median'] = (
        f['feat_price_vs_region_median'].replace([np.inf, -np.inf], np.nan).fillna(1.0)
    )

    f['feat_price_rank_in_region'] = [
        percentile_rank_within_group(p, region_sorted_price_map.get(rid))
        for p, rid in zip(f['price'], f['region_id'])
    ]

    # -------------------------
    # 텍스트 Feature
    # -------------------------
    f['feat_title_length'] = f['title'].str.len()
    f['feat_content_length'] = f['content'].str.len()
    f['feat_title_word_count'] = f['title'].apply(count_words)
    f['feat_content_word_count'] = f['content'].apply(count_words)

    # title_has_brand: top30 브랜드 기준
    f['feat_title_has_brand'] = f['title'].apply(
        lambda x: contains_any_keyword(str(x).lower(), top30_brands)
    )
    f['feat_title_has_condition'] = f['title'].apply(
        lambda x: contains_any_keyword(x, condition_keywords)
    )

    # -------------------------
    # 브랜드 Feature
    # -------------------------
    f['feat_brandName'] = f['brandName'].apply(
        lambda x: x if x in top30_brands_set else 'other'
    )
    f['feat_brand_known'] = (f['brandName'] != 'unknown').astype(int)

    # -------------------------
    # 판매자 Feature
    # -------------------------
    f['feat_sellerTemperature'] = f['sellerTemperature']
    f['feat_seller_temp_bucket'] = f['sellerTemperature'].apply(bucketize_seller_temp)
    f['feat_seller_is_high_trust'] = (f['sellerTemperature'] > 50).astype(int)
    f['feat_seller_is_low_trust'] = (f['sellerTemperature'] < 35).astype(int)

    # -------------------------
    # 이미지 Feature
    # -------------------------
    f['feat_imageCount'] = f['imageCount'].fillna(0)
    f['feat_has_multiple_images'] = (f['feat_imageCount'] >= 3).astype(int)

    f['feat_label'] = f['label'].fillna('unknown')
    f['feat_coarse_label'] = f['coarse_label'].fillna('unknown')
    f['feat_label_conf'] = f['label_conf'].fillna(f['label_conf'].median())

    # -------------------------
    # top3 Feature
    # -------------------------
    f['feat_top1_label'] = f['top3_labels'].apply(parse_top1_label)
    f['feat_top1_score'] = f['top3_scores'].apply(parse_top1_score)
    f['feat_top1_top2_gap'] = f['top3_scores'].apply(parse_top1_top2_gap)

    f['feat_top1_label'] = f['feat_top1_label'].fillna('unknown')
    f['feat_top1_score'] = f['feat_top1_score'].fillna(f['feat_top1_score'].median())
    f['feat_top1_top2_gap'] = f['feat_top1_top2_gap'].fillna(
        f['feat_top1_top2_gap'].median()
    )

    # -------------------------
    # 지역 Feature
    # -------------------------
    f['feat_region_id'] = f['region_id']
    f['feat_region_post_count'] = (
        f['region_id'].map(region_post_count_map).fillna(global_region_post_count)
    )
    f['feat_region_sales_rate'] = (
        f['region_id'].map(region_sales_rate_map).fillna(global_region_sales_rate)
    )
    f['feat_region_price_median'] = (
        f['region_id'].map(region_price_median_map).fillna(global_region_price_median)
    )

    # -------------------------
    # 시간 Feature
    # -------------------------
    f['feat_upload_hour'] = f['createdAt'].dt.hour.fillna(-1).astype(int)
    f['feat_upload_day'] = f['createdAt'].dt.dayofweek.fillna(-1).astype(int)
    f['feat_is_weekend'] = f['feat_upload_day'].isin([5, 6]).astype(int)
    f['feat_is_evening_upload'] = f['feat_upload_hour'].between(18, 23).astype(int)

    # 임시 컬럼 제거
    f.drop(columns=['tmp_label_median_price', 'tmp_region_price_median'], inplace=True)

    return f


# =========================================================
# 7. 각 split에 feature 적용
# =========================================================
train_feat = transform_features(train_df)
valid_feat = transform_features(valid_df)
test_feat = transform_features(test_df)

print('[INFO] Train feature shape:', train_feat.shape)
print('[INFO] Valid feature shape:', valid_feat.shape)
print('[INFO] Test feature shape :', test_feat.shape)


# =========================================================
# 8. 최종 저장 컬럼 선택
# =========================================================
final_feature_cols = [
    # 가격
    'feat_price',
    'feat_log_price',
    'feat_price_bucket',
    'feat_price_vs_label_median',
    'feat_price_vs_region_median',
    'feat_price_rank_in_region',
    # 텍스트
    'feat_title_length',
    'feat_content_length',
    'feat_title_word_count',
    'feat_content_word_count',
    'feat_title_has_brand',
    'feat_title_has_condition',
    # 브랜드
    'feat_brandName',
    'feat_brand_known',
    # 판매자
    'feat_sellerTemperature',
    'feat_seller_temp_bucket',
    'feat_seller_is_high_trust',
    'feat_seller_is_low_trust',
    # 이미지
    'feat_imageCount',
    'feat_has_multiple_images',
    'feat_label',
    'feat_coarse_label',
    'feat_label_conf',
    # top3
    'feat_top1_label',
    'feat_top1_score',
    'feat_top1_top2_gap',
    # 지역
    'feat_region_id',
    'feat_region_post_count',
    'feat_region_sales_rate',
    'feat_region_price_median',
    # 시간
    'feat_upload_hour',
    'feat_upload_day',
    'feat_is_weekend',
    'feat_is_evening_upload',
]

meta_cols = ['id', 'sold']

train_final = train_feat[meta_cols + final_feature_cols].copy()
valid_final = valid_feat[meta_cols + final_feature_cols].copy()
test_final = test_feat[meta_cols + final_feature_cols].copy()

print('[INFO] Number of final features:', len(final_feature_cols))
print('[INFO] Train final shape:', train_final.shape)
print('[INFO] Valid final shape:', valid_final.shape)
print('[INFO] Test final shape :', test_final.shape)


# =========================================================
# 9. 저장
# =========================================================
save_with_parquet_fallback(train_final, output_train_csv, output_train_parquet)
save_with_parquet_fallback(valid_final, output_valid_csv, output_valid_parquet)
save_with_parquet_fallback(test_final, output_test_csv, output_test_parquet)

print('[INFO] All done.')

[INFO] Loading data...
[INFO] Raw shape: (212752, 43)
[INFO] After basic cleaning: (212556, 44)
[INFO] Sold ratio: 0.317695101526186
[INFO] Train shape: (170044, 44)
[INFO] Valid shape: (21256, 44)
[INFO] Test shape : (21256, 44)
[INFO] Top 30 brands: ['lee', 'polo ralph lauren', 'uniqlo', 'the north face', 'nike', 'zara', 'burberry', 'adidas', 'cos', "arc'teryx", 'moncler', 'h&m', 'new balance', 'louis vuitton', "levi's", 'patagonia', 'under armour', 'stone island', 'gap', 'lacoste', '8seconds', 'massimo dutti', 'prada', 'hermes', 'topten', 'dior', 'thom browne', 'gu', 'stussy']
[INFO] Train feature shape: (170044, 78)
[INFO] Valid feature shape: (21256, 78)
[INFO] Test feature shape : (21256, 78)
[INFO] Number of final features: 34
[INFO] Train final shape: (170044, 36)
[INFO] Valid final shape: (21256, 36)
[INFO] Test final shape : (21256, 36)
[INFO] CSV saved -> C:/potenup3/pj02_Daangn-marke/data/processed\train_feature_v21.csv
[INFO] Parquet saved -> C:/potenup3/pj02_Daangn-marke/